In [1]:
import numpy as np
import dask.array as da
from dask.array.image import imread

filename_pattern = r"../../images/AllSkyImages/2010-08/*.JPG"
images = imread(filename_pattern)
images

dask.array<imread, shape=(8595, 480, 640, 3), dtype=uint8, chunksize=(1, 480, 640, 3), chunktype=numpy.ndarray>

In [2]:
channels = images[0].transpose(2, 0, 1)
channels[0]

dask.array<getitem, shape=(480, 640), dtype=uint8, chunksize=(480, 640), chunktype=numpy.ndarray>

In [3]:
red = images.transpose(0, 3, 1, 2)[0][0]
red

dask.array<getitem, shape=(480, 640), dtype=uint8, chunksize=(480, 640), chunktype=numpy.ndarray>

In [4]:
from dask_image import ndfilters

In [5]:
ndfilters.correlate(red, np.ones((3, 3))).compute()

array([[64, 68, 71, ..., 54, 54, 54],
       [74, 71, 69, ..., 54, 54, 54],
       [70, 73, 71, ..., 54, 54, 54],
       ...,
       [16, 16, 22, ...,  9, 10, 11],
       [17, 14, 22, ...,  9,  8,  7],
       [14, 14, 21, ...,  9,  7,  5]], shape=(480, 640), dtype=uint8)

Okay I think I see where this is going, let's see if I can test this with the real atlas.

This first image should definitely have the number 2 in it once and only once. Well twice if you include the bottom bit.

I'm super curious how the correlation will look.

In [6]:
from allsky.classifiers import char_atlas, char_width, char_glyphs, classify_at_cursor

char_two = char_atlas[:, 2 * char_width : 3 * char_width]
char_two

dask.array<getitem, shape=(8, 6), dtype=bool, chunksize=(8, 6), chunktype=numpy.ndarray>

In [7]:
ndfilters.correlate(red, char_two * 255, "constant").compute()

array([[236, 218, 200, ..., 208, 208, 214],
       [225, 203, 189, ..., 204, 205, 217],
       [215, 203, 187, ..., 198, 199, 218],
       ...,
       [239, 238, 235, ..., 245, 245, 249],
       [246, 247, 235, ..., 246, 246, 250],
       [245, 242, 240, ..., 248, 247, 252]], shape=(480, 640), dtype=uint8)

In [8]:
template = char_two.astype(np.float32)
template = template - template.mean()

date_strip = red[6:14, 5:59].astype(np.float32)

ndfilters.correlate(red.astype(np.float32), template, "constant").compute()

array([[-9.6875, -0.4375,  5.6875, ...,  3.    , 10.5   , 12.    ],
       [-6.8125,  3.    ,  2.    , ..., -3.3125,  5.0625,  2.4375],
       [-4.    , -6.375 , -7.875 , ..., -7.625 ,  2.625 , -5.125 ],
       ...,
       [ 3.5625, -2.3125, -4.625 , ..., -2.75  ,  0.0625, -1.75  ],
       [ 0.    , -7.25  ,  0.375 , ..., -1.875 ,  0.625 , -1.5   ],
       [ 2.5625,  0.5625, -1.1875, ..., -1.6875,  1.1875, -2.25  ]],
      shape=(480, 640), dtype=float32)

In [9]:
char_atlas

dask.array<getitem, shape=(8, 66), dtype=bool, chunksize=(8, 66), chunktype=numpy.ndarray>

In [10]:
chars = char_atlas.shape[1] // char_width

# Reshape into (8, chars, char_width)
char_atlas_chars = (
    char_atlas.reshape(8, chars, char_width)
    .transpose(1, 0, 2)
    .astype(np.float32)
    .compute()
)

char_atlas_chars

array([[[0., 1., 1., 1., 0., 0.],
        [1., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 1., 0.],
        [0., 1., 1., 1., 0., 0.]],

       [[0., 0., 1., 0., 0., 0.],
        [0., 1., 1., 0., 0., 0.],
        [1., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.]],

       [[0., 1., 1., 1., 0., 0.],
        [1., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0.]],

       [[0., 1., 1., 1., 0., 0.],
        [1., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 0., 1., 0.],
        

In [11]:
best_char, best_score, scores = classify_at_cursor(
    image=red.astype(np.float32),
    x=5 + char_width * 1,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=6,
    height=8,
)
best_char, best_score, scores

('0',
 0.9998062252998352,
 {'0': 0.9998062252998352,
  '1': -0.08059922605752945,
  '2': 0.40603744983673096,
  '3': 0.681036114692688,
  '4': -0.09153787791728973,
  '5': 0.5023139119148254,
  '6': 0.7777053713798523,
  '7': -0.05396760627627373,
  '8': 0.7773357033729553,
  '9': 0.780293345451355,
  's': 0.20348623394966125})

In [12]:
images_2019 = imread(r"../../images/AllSkyImages/2019-09/*.JPG")
red_2019 = images_2019.transpose(0, 3, 1, 2)[0][0]

best_char, best_score, scores = classify_at_cursor(
    image=red_2019.astype(np.float32),
    x=5 + (char_width) * 1,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=6,
    height=8,
)
best_char, best_score, scores

('5',
 0.36639389395713806,
 {'0': 0.33899861574172974,
  '1': -0.06114104762673378,
  '2': 0.2295859158039093,
  '3': 0.23315724730491638,
  '4': -0.165544331073761,
  '5': 0.36639389395713806,
  '6': 0.29352709650993347,
  '7': 0.020206689834594727,
  '8': 0.2833721339702606,
  '9': 0.25677579641342163,
  's': 0.02235008217394352})

In [13]:
blurzero = (
    ndfilters.gaussian(
        da.from_array(char_glyphs["0"]), (0.0, 0.75), mode="constant"
    ).compute()
    + char_glyphs["0"]
)
blurzero

array([[0.23404631, 1.7657752 , 1.9692547 , 1.7657752 , 0.23404631,
        0.01537264],
       [1.5319073 , 0.2188521 , 0.0303884 , 0.2188521 , 1.5319073 ,
        0.21867366],
       [1.5319073 , 0.2188521 , 0.0303884 , 0.2188521 , 1.5319073 ,
        0.21867366],
       [1.5319073 , 0.2188521 , 0.0303884 , 0.2188521 , 1.5319073 ,
        0.21867366],
       [1.5319073 , 0.2188521 , 0.0303884 , 0.2188521 , 1.5319073 ,
        0.21867366],
       [1.5319073 , 0.2188521 , 0.0303884 , 0.2188521 , 1.5319073 ,
        0.21867366],
       [1.5319073 , 0.2188521 , 0.0303884 , 0.2188521 , 1.5319073 ,
        0.21867366],
       [0.23404631, 1.7657752 , 1.9692547 , 1.7657752 , 0.23404631,
        0.01537264]], dtype=float32)

In [14]:
best_char, best_score, scores = classify_at_cursor(
    image=red_2019.astype(np.float32),
    x=5 + (char_width) * 1,
    y=6,
    atlas={"0": blurzero},
    atlas_chars=["0"],
    width=6,
    height=8,
)
best_char, best_score, scores

('0', 0.4128374457359314, {'0': 0.4128374457359314})